In [0]:
%pip install azure-storage-file-datalake azure-identity pandas pyarrow


In [0]:
dbutils.library.restartPython()

In [0]:
import azure.identity
import azure.storage.filedatalake

print("O SDK da Azure foi encontrado e carregado com sucesso!")

In [0]:
from dotenv import load_dotenv
import os

load_dotenv('.env')

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")

print("client_id carregado:", client_id is not None)
print("tenant_id carregado:", tenant_id is not None)
print("client_secret carregado:", client_secret is not None)
print("storage_account carregado:", storage_account is not None)

In [0]:
# Configurações OAuth do Service Principal passadas a cada chamada do Spark
# Isso garante que a autenticação ocorra diretamente nos executores do cluster
adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

print("Dicionário adls_options preparado para injeção no spark.read.")

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType
from pyspark.sql import functions as F

# 1. Schema explícito com LongType para contornar o TIMESTAMP(NANOS) no Serverless
schema_clientes = StructType([
    StructField("id_cliente", LongType(), True),
    StructField("uuid_cliente", StringType(), True),
    StructField("nome", StringType(), True),
    StructField("sobrenome", StringType(), True),
    StructField("email", StringType(), True),
    StructField("senha_hash", StringType(), True),
    StructField("dt_cadastro", LongType(), True),
    StructField("dt_ultima_atualizacao", LongType(), True)
])

container_name = "raw"
tabela_alvo = "ecommerce_clientes"

file_path = f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/real-time-data/*/*/*/*/{tabela_alvo}.parquet"

print(f"Iniciando a leitura distribuída com schema explícito em:\n{file_path}\n")

try:
    # 2. Leitura com injeção de OAuth e Schema Explícito (funciona no Databricks Serverless)
    df_clientes_raw = (spark.read.options(**adls_options).schema(schema_clientes).parquet(file_path))

    # 3. Conversão paralela: nanossegundos (/ 1000) -> microssegundos -> TimestampType nativo
    df_clientes = (
        df_clientes_raw
        .withColumn(
            "dt_cadastro",
            F.timestamp_micros((F.col("dt_cadastro") / 1000).cast("long"))
        )
        .withColumn(
            "dt_ultima_atualizacao",
            F.timestamp_micros((F.col("dt_ultima_atualizacao") / 1000).cast("long"))
        )
    )

    print("✅ Conexão e leitura concluídas com sucesso no Serverless!")
    print(f"Total de registros: {df_clientes.count()}")
    
    print("\nSchema final tratado:")
    df_clientes.printSchema()

    display(df_clientes.limit(20))

except Exception as e:
    print(f"❌ Erro durante a leitura:\n{e}")